# Heritage Short: Wan 2.2 TI2V-5B image-to-video (Kaggle)

Turns each keyframe into a ~5 s clip. **Not run on the author's machine (no GPU); first run on Kaggle.**

1. On your machine: `make kaggle-export` → `kaggle/heritage_job.zip`.
2. Kaggle → *Datasets → New dataset* → upload the zip (name it `heritage-job`).
3. This notebook → *Add input* → that dataset. *Settings*: Accelerator **GPU T4 x2**, Internet **on**.
4. Run all. Clips are saved one by one, so a timed-out session keeps finished clips; re-running skips them.
5. Download `/kaggle/working/heritage_results.zip` → on your machine `make import ZIP=~/Downloads/heritage_results.zip`.

Budget: a T4 has no bf16 and 16 GB VRAM, so the model runs in fp16 with CPU offload at ~480p.
Expect several minutes per clip; `MAX_CLIPS` limits a session. Shots marked `"video": "wan"` come first.

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
import glob, json, os, socket, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

JOB = Path(glob.glob("/kaggle/input/**/jobs.json", recursive=True)[0]).parent \
    if glob.glob("/kaggle/input/**/jobs.json", recursive=True) else Path("job")
jobs = json.loads((JOB / "jobs.json").read_text())
OUT = Path("/kaggle/working/results") if Path("/kaggle/working").exists() else Path("results")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "runs_kaggle.jsonl"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-kaggle"

def gpu_name():
    try:
        import torch
        return torch.cuda.get_device_name(0)
    except Exception:
        return "unknown"

def log_call(stage, shot, model, params, seed, latency_s, status, output=None, error=None):
    """Same shape as pipeline/common.py log lines, so make import can merge them."""
    rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "run_id": RUN_ID,
           "event": "model_call", "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "output": output, "attempt": 1, "status": status,
           "latency_s": round(latency_s, 2), "gpu": gpu_name(),
           "cost": {"gpu_minutes": round(latency_s / 60, 2), "tier": "free (Kaggle weekly GPU quota)"}}
    if error:
        rec["error"] = str(error)[:300]
    with LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{status:5} {shot or ''} {latency_s:7.1f}s {output or error or ''}")

print("job folder:", JOB, "| storyboard version", jobs["storyboard_version"])

In [ ]:
import numpy as np, torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

P = jobs["wan_params"]
MAX_CLIPS = 4            # raise if your quota allows
STEPS = P["steps"]       # 30 for drafts; the model card uses 50
DTYPE = torch.float16    # T4 has no native bf16. If clips come out black (NaN), try torch.bfloat16.

vae = AutoencoderKLWan.from_pretrained(P["model"], subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(P["model"], vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()   # fits the 5B model + UMT5 text encoder on a 16 GB T4
mod = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
print("loaded", P["model"], "mod", mod, "gpu", gpu_name())

In [ ]:
(OUT / "clips").mkdir(exist_ok=True)
done = 0
for job in jobs["wan"]:
    out = OUT / "clips" / job["out"]
    if out.exists():
        print("skip (done)", job["shot"]); continue
    if done >= MAX_CLIPS:
        break
    image = load_image(str(JOB / job["keyframe"]))
    ratio = image.height / image.width
    h = round(np.sqrt(P["max_area"] * ratio)) // mod * mod
    w = round(np.sqrt(P["max_area"] / ratio)) // mod * mod
    image = image.resize((w, h))
    params = {"width": w, "height": h, "num_frames": P["num_frames"], "steps": STEPS,
              "guidance_scale": P["guidance_scale"], "dtype": str(DTYPE), "prompt": job["prompt"]}
    t0 = time.perf_counter()
    try:
        frames = pipe(image=image, prompt=job["prompt"], negative_prompt=P["negative_prompt"],
                      height=h, width=w, num_frames=P["num_frames"], guidance_scale=P["guidance_scale"],
                      num_inference_steps=STEPS,
                      generator=torch.Generator("cuda").manual_seed(job["seed"])).frames[0]
        export_to_video(frames, str(out), fps=P["fps"])
        log_call("shots", job["shot"], P["model"], params, job["seed"], time.perf_counter() - t0, "ok",
                 output=f"outputs/shots/wan/{job['out']}")
        done += 1
    except Exception as err:   # OOM or NaN: log it and move on; Ken Burns covers this shot
        log_call("shots", job["shot"], P["model"], params, job["seed"], time.perf_counter() - t0, "error", error=err)
        torch.cuda.empty_cache()

In [ ]:
# Zip everything in results/ so one download brings it home: then `make import ZIP=...`
zpath = OUT.parent / "heritage_results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
print("download:", zpath, round(zpath.stat().st_size / 1e6, 1), "MB")